# Task 2: Rainfall Prediction with Apache Spark MLlib (Baseline Model)

### Project Task Description
Build a machine learning model that estimates the **probability that a monitoring station will accumulate more than a given amount of rainfall in the next 30 minutes**. 
The baseline model takes the following inputs:
1. **Station** (`station_id` indexed, coordinates: `location_longitude`, `location_latitude`)
2. **Prediction Time** (Temporal features: hour, month, cyclical sin/cos encodings, monsoon season)
3. **Current Rainfall Reading** (`reading_value` at prediction time $t$)
4. **Rainfall Threshold** ($T$, e.g., $0.00\text{ mm}$ (occurrence benchmark), $0.20\text{ mm}$, $0.40\text{ mm}$, $1.00\text{ mm}$, $3.00\text{ mm}$, $24.20\text{ mm}$)

### Workflow Overview:
1. **Spark Session Initialization**: Tuned memory allocation and shuffle partition settings.
2. **Dataset Reload & Streamlined Preprocessing**: Directly prune redundant/constant columns justified in Part 1 (`reading_type`, `reading_unit`, `station_device_id`, `station_name`, `update_timestamp`, `reading_update_timestamp`, `date`).
3. **Timestamp Decomposition & Temporal Features**: Extract hour, month, cyclical sin/cos time-of-day and seasonal encodings, and monsoon season.
4. **Target Construction (30-Minute Forward Window)**: Calculate ground-truth accumulated rainfall in $[t+1\text{s}, t+1860\text{s}]$ and filter for complete 6-reading windows.
5. **Chronological Train/Test Split**: Split into Train (2017–2022) and Test (2023–2024) before sampling to prevent data leakage and preserve test integrity.
6. **Stratified Train Sampling & Threshold Explosion**:
   - **Training Set**: Apply stratified downsampling to balance active rain vs. dry periods.
   - **Testing Set**: Preserve the true, unmanipulated real-world prior distribution (94% dry / 6% rain).
   - **Explosion**: Replicate both sets across the 6 candidate thresholds (including 0.00 mm occurrence benchmark) so threshold is an active numerical input feature with target strictly $> T$.
7. **MLlib Pipeline & Model Training**: Build a baseline Spark MLlib Pipeline (`StringIndexer`, `OneHotEncoder`, `VectorAssembler`, `StandardScaler`, and `LogisticRegression`).
8. **Model Evaluation & Inference Demonstration**: Evaluate overall and per-threshold test performance (AUC-ROC, AUC-PR), benchmark against persistence and random baselines at $T=0$, and verify monotonic probability estimation.


In [1]:
from pyspark.sql import SparkSession

# Initialize Spark session with tuned memory for MLlib operations
spark = (
    SparkSession.builder
    .appName("Singapore_Rainfall_Prediction_Task2")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print("SparkSession initialized successfully!")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/11 12:28:51 WARN Utils: Your hostname, Corneliuss-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.31.244 instead (on interface en0)
26/10/11 12:28:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/corneliusaxeltjandra/Documents/Distributed DB/Project2/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/11 12:28:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applic

Spark version: 4.2.0
SparkSession initialized successfully!


In [2]:
import glob
import os
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, TimestampType

# 1. Locate all CSV datasets (2017 to 2024)
dataset_dir = "Dataset"
csv_files = sorted(glob.glob(os.path.join(dataset_dir, "*.csv")))
print(f"Found {len(csv_files)} dataset files to load:")
for f in csv_files:
    print(f" - {f}")

# 2. Reload datasets into Spark DataFrame
df_raw = spark.read.csv(csv_files, header=True, inferSchema=True)

# 3. Prune columns justified as redundant / irrelevant in Part 1:
# - reading_type, reading_unit: 100% constant zero-variance values
# - station_device_id: 100% identical mapping to station_id
# - station_name: Redundant and has name collision (e.g. Upper Changi Road North)
# - update_timestamp, reading_update_timestamp: Database sync metadata, irrelevant to prediction
# - date: Redundant with timestamp
cols_to_drop = [
    "reading_type", 
    "reading_unit", 
    "station_device_id", 
    "station_name", 
    "update_timestamp", 
    "reading_update_timestamp", 
    "date"
]

df = df_raw.drop(*cols_to_drop)

# Cast types cleanly and drop missing records
df = (df
    .withColumn("timestamp", F.col("timestamp").cast(TimestampType()))
    .withColumn("reading_value", F.col("reading_value").cast(DoubleType()))
    .withColumn("location_longitude", F.col("location_longitude").cast(DoubleType()))
    .withColumn("location_latitude", F.col("location_latitude").cast(DoubleType()))
    .dropna(subset=["timestamp", "station_id", "reading_value"])
)

print(f"\nProcessed DataFrame Schema:")
df.printSchema()
print(f"Sample Records:")
df.show(5, truncate=False)


Found 8 dataset files to load:
 - Dataset/d_024fb501ce7092b71bb713eaf54fa7eb.csv
 - Dataset/d_1990a5a1aeaf3dd243cf4dae294a61c4.csv
 - Dataset/d_3b41598f74f1f11fc3430348fea51af5.csv
 - Dataset/d_42d64cc6c176ace1c52fbb40b9ede302.csv
 - Dataset/d_61995f092320e7155b7528050880b502.csv
 - Dataset/d_9e7de44094f876f6804b8b5bcee45c81.csv
 - Dataset/d_a0b69d3e02576a1fd0ab673e71f83507.csv
 - Dataset/d_f864cc30d58b467db83659ad17c737bf.csv



Processed DataFrame Schema:
root
 |-- timestamp: timestamp (nullable = true)
 |-- station_id: string (nullable = true)
 |-- location_longitude: double (nullable = true)
 |-- location_latitude: double (nullable = true)
 |-- reading_value: double (nullable = true)

Sample Records:
+-------------------+----------+------------------+-----------------+-------------+
|timestamp          |station_id|location_longitude|location_latitude|reading_value|
+-------------------+----------+------------------+-----------------+-------------+
|2018-01-01 00:00:00|S77       |103.8125          |1.2937           |0.0          |
|2018-01-01 00:00:00|S109      |103.8492          |1.3764           |0.4          |
|2018-01-01 00:00:00|S117      |103.679           |1.256            |1.2          |
|2018-01-01 00:00:00|S55       |103.886           |1.3837           |0.2          |
|2018-01-01 00:00:00|S64       |103.7603          |1.3824           |0.0          |
+-------------------+----------+---------------

In [3]:
from pyspark.sql import functions as F
import numpy as np

# 1. Decompose timestamp into core temporal attributes
df = (df
    .withColumn("year", F.year("timestamp"))
    .withColumn("month", F.month("timestamp"))
    .withColumn("day", F.dayofmonth("timestamp"))
    .withColumn("hour", F.hour("timestamp"))
    .withColumn("minute", F.minute("timestamp"))
    .withColumn("day_of_week", F.dayofweek("timestamp"))
    .withColumn("day_of_year", F.dayofyear("timestamp"))
)

# 2. Add continuous minute-of-day feature: minute_of_day = hour * 60 + minute
df = df.withColumn("minute_of_day", F.col("hour") * 60 + F.col("minute"))

# 3. Add Cyclical Encodings (sin and cos) for diurnal (24h) and annual (seasonal) cycles:
# Hour cycle (1440 minutes in a day): Ensures 23:55 is adjacent to 00:00
pi_val = float(np.pi)
df = (df
    .withColumn("sin_time_of_day", F.sin(2 * pi_val * F.col("minute_of_day") / 1440.0))
    .withColumn("cos_time_of_day", F.cos(2 * pi_val * F.col("minute_of_day") / 1440.0))
    .withColumn("sin_month", F.sin(2 * pi_val * F.col("month") / 12.0))
    .withColumn("cos_month", F.cos(2 * pi_val * F.col("month") / 12.0))
)

# 4. Singapore Monsoon Season Category:
# 1: Northeast Monsoon Wet (Dec-Jan), 2: NE Monsoon Dry (Feb), 3: Inter-monsoon 1 (Mar-May),
# 4: Southwest Monsoon (Jun-Sep), 5: Inter-monsoon 2 (Oct-Nov)
df = df.withColumn(
    "monsoon_season",
    F.when(F.col("month").isin(12, 1), 1)
     .when(F.col("month") == 2, 2)
     .when(F.col("month").isin(3, 4, 5), 3)
     .when(F.col("month").isin(6, 7, 8, 9), 4)
     .otherwise(5)
)

print("Temporal feature extraction complete!")
df.select("timestamp", "station_id", "reading_value", "hour", "sin_time_of_day", "cos_time_of_day", "monsoon_season").show(5)


Temporal feature extraction complete!
+-------------------+----------+-------------+----+---------------+---------------+--------------+
|          timestamp|station_id|reading_value|hour|sin_time_of_day|cos_time_of_day|monsoon_season|
+-------------------+----------+-------------+----+---------------+---------------+--------------+
|2018-01-01 00:00:00|       S77|          0.0|   0|            0.0|            1.0|             1|
|2018-01-01 00:00:00|      S109|          0.4|   0|            0.0|            1.0|             1|
|2018-01-01 00:00:00|      S117|          1.2|   0|            0.0|            1.0|             1|
|2018-01-01 00:00:00|       S55|          0.2|   0|            0.0|            1.0|             1|
|2018-01-01 00:00:00|       S64|          0.0|   0|            0.0|            1.0|             1|
+-------------------+----------+-------------+----+---------------+---------------+--------------+
only showing top 5 rows


In [4]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

# === Forward 30-Minute Target Construction ===
# Readings arrive at 5-minute intervals (300s).
# A complete forward 30-minute window expects exactly 6 readings.
# Range [1, 1860] (30 mins + 60s jitter tolerance) captures [t+5m, ..., t+30m].

w_forward_30m = (
    Window.partitionBy("station_id")
    .orderBy(F.col("timestamp").cast("long"))
    .rangeBetween(1, 1860)
)

df = (df
    .withColumn("next_30m_reading_count", F.count("reading_value").over(w_forward_30m))
    .withColumn("next_30m_rainfall", F.round(F.sum("reading_value").over(w_forward_30m), 2))
)

# Filter for complete windows (exactly 6 future readings) to eliminate sensor dropouts and tail ends
df_clean = df.filter(F.col("next_30m_reading_count") == 6)

print("Filtered to complete 30-minute forward windows:")
df_clean.select("station_id", "timestamp", "reading_value", "next_30m_reading_count", "next_30m_rainfall").show(5)


Filtered to complete 30-minute forward windows:


+----------+-------------------+-------------+----------------------+-----------------+
|station_id|          timestamp|reading_value|next_30m_reading_count|next_30m_rainfall|
+----------+-------------------+-------------+----------------------+-----------------+
|       S08|2017-01-01 08:04:59|          0.0|                     6|              0.0|
|       S08|2017-01-01 08:09:59|          0.0|                     6|              0.0|
|       S08|2017-01-01 08:14:59|          0.0|                     6|              0.0|
|       S08|2017-01-01 08:19:59|          0.0|                     6|              0.0|
|       S08|2017-01-01 08:24:59|          0.0|                     6|              0.0|
+----------+-------------------+-------------+----------------------+-----------------+
only showing top 5 rows


In [5]:
from pyspark.sql import functions as F

# === Step 5: Chronological Train / Test Split ===
# To prevent temporal data leakage and protect test integrity, split BEFORE sampling:
#   Raw Training Set: 2017 to 2022 (6 years)
#   Raw Testing Set:  2023 to 2024 (2 years)

raw_train = df_clean.filter(F.col("year") <= 2022)
raw_test  = df_clean.filter(F.col("year") >= 2023)

raw_train_count = raw_train.count()
raw_test_count  = raw_test.count()
total_raw = raw_train_count + raw_test_count

print(f"=== Chronological Split (Prior to Sampling) ===")
print(f"Raw Train Set (2017-2022): {raw_train_count:>10,d} rows ({raw_train_count / total_raw * 100:.1f}%)")
print(f"Raw Test Set  (2023-2024): {raw_test_count:>10,d} rows ({raw_test_count / total_raw * 100:.1f}%)")


=== Chronological Split (Prior to Sampling) ===
Raw Train Set (2017-2022): 34,542,404 rows (72.5%)
Raw Test Set  (2023-2024): 13,094,128 rows (27.5%)


In [6]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark import StorageLevel

# === 1. Candidate Thresholds Grounded in Part 1 Empirical Percentiles ===
# Benchmark: 0.00 mm (Occurrence) | 20th: 0.20 mm | 40th: 0.40 mm | 60th: 1.00 mm | 80th: 3.00 mm | 99th: 24.20 mm
THRESHOLDS = [0.00, 0.20, 0.40, 1.00, 3.00, 24.20]
threshold_array = F.array([F.lit(t) for t in THRESHOLDS])

# === 2. Stratified Subsampling on TRAINING Set Only ===
# Downsample majority dry periods while retaining the vast majority of active rain events
TRAIN_FRACTIONS = {
    0: 0.08,  # 8% of dry periods (0.0 mm)
    1: 0.80   # 80% of active rain periods (> 0.0 mm)
}

train_tagged = raw_train.withColumn("is_rain", (F.col("next_30m_rainfall") > 0.0).cast("int"))
train_sampled = (
    train_tagged.sampleBy("is_rain", fractions=TRAIN_FRACTIONS, seed=42)
    .drop("is_rain")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

# === 3. Natural-Distribution Testing Set (Full 2023-2024 Evaluation) ===
# Use 100% of raw_test (13,094,128 rows) to evaluate on the full test set across all 6 thresholds
TEST_SAMPLE_FRACTION = 1.0
test_sampled = (
    raw_test if TEST_SAMPLE_FRACTION == 1.0 else
    raw_test.sample(withReplacement=False, fraction=TEST_SAMPLE_FRACTION, seed=42)
).persist(StorageLevel.MEMORY_AND_DISK)

# Materialize sampled sets once to sever the 48M-row Window shuffle lineage
train_sampled.count()
test_sampled.count()

# === 4. Explode Both Sets Across Candidate Thresholds ===
# Each row is replicated 6x (once per threshold T).
# Target label: 1.0 if future 30m rainfall > threshold, else 0.0 (strictly greater than)
train_df = (
    train_sampled
    .withColumn("rainfall_threshold", F.explode(threshold_array))
    .withColumn("label", (F.col("next_30m_rainfall") > F.col("rainfall_threshold")).cast(DoubleType()))
)

test_df = (
    test_sampled
    .withColumn("rainfall_threshold", F.explode(threshold_array))
    .withColumn("label", (F.col("next_30m_rainfall") > F.col("rainfall_threshold")).cast(DoubleType()))
)

# === 5. Verify Class Distribution Comparison ===
train_final_count = train_df.count()
test_final_count = test_df.count()

print(f"=== Exploded Dataset Sizes (6x Thresholds: {THRESHOLDS}) ===")
print(f"Final Train Set (Stratified & Exploded 6x): {train_final_count:>10,d} rows")
print(f"Final Test Set  (Natural & Exploded 6x):    {test_final_count:>10,d} rows")

print(f"\n=== Class Breakdown Comparison at Threshold > 0.00 mm (Occurrence Benchmark) ===")
train_pos_0 = train_df.filter((F.col("rainfall_threshold") == 0.00) & (F.col("label") == 1.0)).count()
train_total_0 = train_df.filter(F.col("rainfall_threshold") == 0.00).count()
test_pos_0 = test_df.filter((F.col("rainfall_threshold") == 0.00) & (F.col("label") == 1.0)).count()
test_total_0 = test_df.filter(F.col("rainfall_threshold") == 0.00).count()

print(f"Train Set (Balanced for Learning): {train_pos_0:>8,d} / {train_total_0:>8,d} ({train_pos_0 / train_total_0 * 100:>5.1f}% positive)")
print(f"Test Set  (Real-World Base Rate):  {test_pos_0:>8,d} / {test_total_0:>8,d} ({test_pos_0 / test_total_0 * 100:>5.1f}% positive)")

print(f"\n=== Class Breakdown Comparison at Threshold > 0.20 mm ===")
train_pos_20 = train_df.filter((F.col("rainfall_threshold") == 0.20) & (F.col("label") == 1.0)).count()
train_total_20 = train_df.filter(F.col("rainfall_threshold") == 0.20).count()
test_pos_20 = test_df.filter((F.col("rainfall_threshold") == 0.20) & (F.col("label") == 1.0)).count()
test_total_20 = test_df.filter(F.col("rainfall_threshold") == 0.20).count()

print(f"Train Set (Balanced for Learning): {train_pos_20:>8,d} / {train_total_20:>8,d} ({train_pos_20 / train_total_20 * 100:>5.1f}% positive)")
print(f"Test Set  (Real-World Base Rate):  {test_pos_20:>8,d} / {test_total_20:>8,d} ({test_pos_20 / test_total_20 * 100:>5.1f}% positive)")


=== Exploded Dataset Sizes (6x Thresholds: [0.0, 0.2, 0.4, 1.0, 3.0, 24.2]) ===
Final Train Set (Stratified & Exploded 6x): 25,536,606 rows
Final Test Set  (Natural & Exploded 6x):    78,564,768 rows

=== Class Breakdown Comparison at Threshold > 0.00 mm (Occurrence Benchmark) ===
Train Set (Balanced for Learning): 1,659,982 / 4,256,101 ( 39.0% positive)
Test Set  (Real-World Base Rate):   854,582 / 13,094,128 (  6.5% positive)

=== Class Breakdown Comparison at Threshold > 0.20 mm ===
Train Set (Balanced for Learning): 1,133,496 / 4,256,101 ( 26.6% positive)
Test Set  (Real-World Base Rate):   587,283 / 13,094,128 (  4.5% positive)


In [7]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression

# 1. Index categorical station_id
station_indexer = StringIndexer(
    inputCol="station_id", 
    outputCol="station_idx", 
    handleInvalid="keep"
)

# 2. One-Hot Encode station_idx and monsoon_season (avoids assuming arbitrary ordinal distances)
encoder = OneHotEncoder(
    inputCols=["station_idx", "monsoon_season"],
    outputCols=["station_ohe", "monsoon_ohe"],
    dropLast=False,
    handleInvalid="keep"
)

# 3. Select baseline input features:
# - Station: station_ohe, location_longitude, location_latitude
# - Prediction Time: sin_time_of_day, cos_time_of_day, sin_month, cos_month, monsoon_ohe
#   (Raw hour and month are omitted to avoid collinearity and artificial boundary discontinuities)
# - Current Rainfall: reading_value (instantaneous rain at prediction time)
# - Rainfall Threshold: rainfall_threshold
feature_cols = [
    "station_ohe",
    "location_longitude",
    "location_latitude",
    "sin_time_of_day",
    "cos_time_of_day",
    "sin_month",
    "cos_month",
    "monsoon_ohe",
    "reading_value",
    "rainfall_threshold"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features", handleInvalid="skip")

# 4. Standardize features
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=False)

# 5. Logistic Regression Classifier (provides calibrated probability estimates via sigmoid)
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=50,
    regParam=0.01,
    elasticNetParam=0.1
)

# Assemble into MLlib Pipeline
pipeline = Pipeline(stages=[station_indexer, encoder, assembler, scaler, lr])
print("Baseline MLlib Pipeline successfully assembled with OneHotEncoder!")


Baseline MLlib Pipeline successfully assembled with OneHotEncoder!


In [8]:
from pyspark import StorageLevel

# === Step 8: Train the MLlib Baseline Pipeline ===
print("Persisting training set and training MLlib Logistic Regression model...")
train_df = train_df.persist(StorageLevel.MEMORY_AND_DISK)
train_df.count()  # Materialize once so optimization iterations do not re-read from disk

model = pipeline.fit(train_df)
train_df.unpersist()  # Free memory for evaluation
print("Model training completed successfully!")


Persisting training set and training MLlib Logistic Regression model...


26/10/11 12:31:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/10/11 12:33:39 ERROR Utils: Uncaught exception in thread executor-heartbeater
java.lang.NullPointerException: Cannot invoke "java.util.concurrent.ConcurrentHashMap.isEmpty()" because the return value of "org.apache.spark.sql.util.PartitionKeyedAccumulator.byPartition()" is null
	at org.apache.spark.sql.util.PartitionKeyedAccumulator.isZero(PartitionKeyedAccumulator.scala:51)
	at org.apache.spark.executor.Executor.$anonfun$reportHeartBeat$2(Executor.scala:1536)
	at org.apache.spark.executor.Executor.$anonfun$reportHeartBeat$2$adapted(Executor.scala:1536)
	at scala.collection.immutable.Vector.filterImpl(Vector.scala:166)
	at scala.collection.immutable.Vector.filterImpl(Vector.scala:116)
	at scala.collection.StrictOptimizedIterableOps.filterNot(StrictOptimizedIterableOps.scala:220)
	at scala.collecti

Model training completed successfully!


In [9]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark import StorageLevel

print("Evaluating model predictions on real-world test set (2023-2024)...")

# 1. Project required columns (including metadata needed for demonstration) and persist in memory
eval_cols = [
    "station_id", "timestamp", "reading_value", "next_30m_rainfall",
    "rainfall_threshold", "label", "rawPrediction", "prediction", "probability"
]
predictions = (
    model.transform(test_df)
    .select(eval_cols)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

total_eval_rows = predictions.count()  # Materialize cache once
print(f"Cached {total_eval_rows:,} evaluation rows across all 6 thresholds ({THRESHOLDS}). Computing metrics...")

# 2. Overall Evaluation Metrics across all evaluated thresholds
evaluator_roc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
evaluator_pr  = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderPR")

roc_auc = evaluator_roc.evaluate(predictions)
pr_auc  = evaluator_pr.evaluate(predictions)

print(f"\n=== Overall Model Evaluation on Natural Test Set ===")
print(f"Area Under ROC (AUC-ROC): {roc_auc:.4f}")
print(f"Area Under PR  (AUC-PR):  {pr_auc:.4f}")

# 3. Dedicated Benchmark Comparison at Threshold 0 (> 0.00 mm Rain Occurrence)
# Compares MLlib model vs. Persistence baseline (current rain) vs. Random/Climatology baseline
preds_0 = predictions.filter(F.col("rainfall_threshold") == 0.0)
n_test_0 = preds_0.count()
pos_0 = preds_0.filter(F.col("label") == 1.0).count()
base_rate = (pos_0 / n_test_0) if n_test_0 > 0 else 0.0

model_roc_0 = evaluator_roc.evaluate(preds_0)
model_pr_0  = evaluator_pr.evaluate(preds_0)

# Persistence benchmark: score = instantaneous rainfall reading at prediction time (reading_value)
eval_persist_roc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="reading_value", metricName="areaUnderROC")
eval_persist_pr  = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="reading_value", metricName="areaUnderPR")
persist_roc_0 = eval_persist_roc.evaluate(preds_0)
persist_pr_0  = eval_persist_pr.evaluate(preds_0)

print(f"\n=== Benchmark Comparison at Threshold 0 (> 0.00 mm Rain Occurrence) ===")
print(f"{'Model / Baseline':<28} | {'Test Rows':<10} | {'Base Rate':<11} | {'AUC-ROC':<10} | {'AUC-PR':<10}")
print("-" * 80)
print(f"{'MLlib Baseline Model':<28} | {n_test_0:>9,d} | {base_rate * 100:>8.2f}%   | {model_roc_0:>8.4f} | {model_pr_0:>8.4f}")
print(f"{'Persistence (Current Rain)':<28} | {n_test_0:>9,d} | {base_rate * 100:>8.2f}%   | {persist_roc_0:>8.4f} | {persist_pr_0:>8.4f}")
print(f"{'Climatology / Random Prior':<28} | {n_test_0:>9,d} | {base_rate * 100:>8.2f}%   | {0.5000:>8.4f} | {base_rate:>8.4f}")

# 4. Single-Pass Threshold Statistics Aggregation
thresh_stats = (
    predictions.groupBy("rainfall_threshold")
    .agg(
        F.count("*").alias("test_rows"),
        F.sum(F.when(F.col("label") == 1.0, 1).otherwise(0)).alias("pos_rows"),
        F.countDistinct("label").alias("n_classes")
    )
    .toPandas()
    .set_index("rainfall_threshold")
)

# 5. Per-Threshold Performance Breakdown (strictly > threshold)
print(f"\n=== Performance Breakdown Per Threshold on Natural Test Set (Strictly > Threshold) ===")
print(f"{'Threshold':<20} | {'Test Rows':<10} | {'Positive Rate':<14} | {'AUC-ROC':<10} | {'AUC-PR':<10}")
print("-" * 77)

for t in THRESHOLDS:
    if t in thresh_stats.index:
        t_count = int(thresh_stats.loc[t, "test_rows"])
        pos_count = int(thresh_stats.loc[t, "pos_rows"])
        n_classes = int(thresh_stats.loc[t, "n_classes"])
        pos_rate = (pos_count / t_count) * 100
        label_prefix = f"> {t:>5.2f} mm (BM)" if t == 0.0 else f"> {t:>5.2f} mm     "
        if n_classes > 1:
            t_preds = predictions.filter(F.col("rainfall_threshold") == t)
            t_roc = evaluator_roc.evaluate(t_preds)
            t_pr  = evaluator_pr.evaluate(t_preds)
            print(f"{label_prefix:<20} | {t_count:>9,d} | {pos_rate:>11.2f}%   | {t_roc:>8.4f} | {t_pr:>8.4f}")
        else:
            print(f"{label_prefix:<20} | {t_count:>9,d} | {pos_rate:>11.2f}%   | Single class present")


Evaluating model predictions on real-world test set (2023-2024)...


26/10/11 12:35:53 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 143626 ms exceeds timeout 120000 ms
26/10/11 12:35:53 WARN SparkContext: Killing executors is not supported by current scheduler.


Cached 78,564,768 evaluation rows across all 6 thresholds ([0.0, 0.2, 0.4, 1.0, 3.0, 24.2]). Computing metrics...



=== Overall Model Evaluation on Natural Test Set ===
Area Under ROC (AUC-ROC): 0.8612
Area Under PR  (AUC-PR):  0.4200



=== Benchmark Comparison at Threshold 0 (> 0.00 mm Rain Occurrence) ===
Model / Baseline             | Test Rows  | Base Rate   | AUC-ROC    | AUC-PR    
--------------------------------------------------------------------------------
MLlib Baseline Model         | 13,094,128 |     6.53%   |   0.7686 |   0.4113
Persistence (Current Rain)   | 13,094,128 |     6.53%   |   0.7033 |   0.6609
Climatology / Random Prior   | 13,094,128 |     6.53%   |   0.5000 |   0.0653


...

>  0.20 mm           | 13,094,128 |        4.49%   |   0.8135 |   0.4548


>  0.40 mm           | 13,094,128 |        3.79%   |   0.8273 |   0.4651


>  1.00 mm           | 13,094,128 |        2.60%   |   0.8482 |   0.4686


>  3.00 mm           | 13,094,128 |        1.28%   |   0.8597 |   0.4203


> 24.20 mm           | 13,094,128 |        0.06%   |   0.9131 |   0.1297


In [10]:
from pyspark.sql.types import DoubleType
import pyspark.sql.functions as F

# UDF to extract probability of positive class P(Rain > Threshold) from probability Vector
extract_prob_udf = F.udf(lambda v: float(v[1]) if v is not None else None, DoubleType())

# Ensure demo_source has station_id and timestamp
if "station_id" not in predictions.columns or "timestamp" not in predictions.columns:
    demo_source = model.transform(test_df)
else:
    demo_source = predictions

pred_with_prob = demo_source.withColumn("estimated_prob", extract_prob_udf("probability"))

# Pick a sample rain observation from test set and inspect how probability scales across thresholds
print("=== Demonstration: Estimated Probability Across Thresholds for the Same Observation ===")
sample_event = (
    pred_with_prob.filter((F.col("next_30m_rainfall") > 2.0) & (F.col("label") == 1.0))
    .select("station_id", "timestamp")
    .first()
)

if sample_event:
    sample_station = sample_event["station_id"]
    sample_ts = sample_event["timestamp"]
    
    demo_df = (
        pred_with_prob
        .filter((F.col("station_id") == sample_station) & (F.col("timestamp") == sample_ts))
        .select(
            "station_id",
            "timestamp",
            "reading_value",
            "next_30m_rainfall",
            "rainfall_threshold",
            F.round("estimated_prob", 4).alias("estimated_prob"),
            "prediction",
            "label"
        )
        .orderBy("rainfall_threshold")
    )
    demo_df.show(truncate=False)
else:
    pred_with_prob.select("station_id", "timestamp", "rainfall_threshold", "next_30m_rainfall", F.round("estimated_prob", 4).alias("prob"), "label").show(15)


/Users/corneliusaxeltjandra/Documents/Distributed DB/Project2/.venv/lib/python3.14/site-packages/pyspark/sql/udf.py:116: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/Users/corneliusaxeltjandra/Documents/Distributed DB/Project2/.venv/lib/python3.14/site-packages/pyspark/sql/udf.py:120: RuntimeWarning: Arrow optimization failed to enable because PyArrow or Pandas is not installed. Falling back to a non-Arrow-optimized UDF.
  warnings.warn(


=== Demonstration: Estimated Probability Across Thresholds for the Same Observation ===


+----------+-------------------+-------------+-----------------+------------------+--------------+----------+-----+
|station_id|timestamp          |reading_value|next_30m_rainfall|rainfall_threshold|estimated_prob|prediction|label|
+----------+-------------------+-------------+-----------------+------------------+--------------+----------+-----+
|S60       |2023-01-19 13:10:00|0.0          |3.0              |0.0               |0.2905        |0.0       |1.0  |
|S60       |2023-01-19 13:10:00|0.0          |3.0              |0.2               |0.2841        |0.0       |1.0  |
|S60       |2023-01-19 13:10:00|0.0          |3.0              |0.4               |0.2777        |0.0       |1.0  |
|S60       |2023-01-19 13:10:00|0.0          |3.0              |1.0               |0.2591        |0.0       |1.0  |
|S60       |2023-01-19 13:10:00|0.0          |3.0              |3.0               |0.2031        |0.0       |0.0  |
|S60       |2023-01-19 13:10:00|0.0          |3.0              |24.2    